In [1]:

!pip install langchain langchain-community pypdf langchain-text-splitters --quiet

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('Gita.pdf')
pages = loader.load()
print(f'Loaded {len(pages)} pages')
for page in pages[:2]:
    print(f'Page {page.metadata["page"]}: {len(page.page_content)} chars')

PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Loaded 447 pages
Page 0: 233 chars
Page 1: 463 chars


In [3]:
from langchain_text_splitters  import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800 ,     # max characters per chunk
    chunk_overlap=100,    # overlap between chunks to preserve context
)

chunks = splitter.split_documents(pages)

print(f'Number of chunks: {len(chunks)}')
print()
for i, chunk in enumerate(chunks):
    print(f'--- Chunk {i+1} ({len(chunk.page_content)} chars) ---')
    print(chunk.page_content)
    print()

Number of chunks: 1362

--- Chunk 1 (233 chars) ---
i 
 
 
 
 
The Bhagavad Gita 
Based on HH Sri Raghavendra Teertha’s Gita Vivruti 
& 
Lectures by HH Sri Vidyasagara Madhava Teertha 
 
 
 
Compiled By 
Dr. Giridhar Boray 
 
 
 
 
 
 
 
TIRUMALA TIRUPATI DEVASTHANAMS  
TIRUPATI 
2023

--- Chunk 2 (463 chars) ---
iijkhh 
  
 
 
The Bhagavad Gita 
(Based on HH Sri Raghavendra Teertha’s Gita Vivruti) 
 
 
Compiled By 
Dr. Giridhar Boray 
 
T.T.D. Religious Publications Series No.1451 
 All Rights Reserved 
 
First Edition : 2023 
 
Copies : 500 
 
Published  by : 
Sri A.V. Dharma Reddy, IDES 
Executive Officer, 
Tirumala Tirupati Devasthanams, 
Tirupati. 
 
D.T.P: 
Publications Division, 
T.T.D, Tirupati. 
 
Printed at : 
Tirumala Tirupati Devasthanams Press, 
Tirupati.

--- Chunk 3 (3 chars) ---
iii

--- Chunk 4 (6 chars) ---
ivjkhh

--- Chunk 5 (1 chars) ---
v

--- Chunk 6 (6 chars) ---
vijkhh

--- Chunk 7 (59 chars) ---
vii 
 
 
 
 
 
 
 
 
 
 
HH SRI RAGHAVENDRA TEERTHA SWAMYJI

--- 

In [4]:
!pip install ollama

In [5]:
# Pull the 8B version of LLaMA 3.1
!ollama pull llama3.1:8b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling 667b0c1932bc: 100% ▕██████████████████▏ 4.9 GB                         
pulling 948af2743fc7: 100% ▕██████████████████▏ 1.5 KB                         
pulling 0ba8f0e314b4: 100% ▕██████████████████▏  12 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 455f34728c9b: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 


In [6]:
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model='llama3.1:8b')

# Create vector store from documents
# This embeds all documents and stores them in Chroma
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory='./chroma_db'  # save to disk
)

print(f'Vector store created with {vectorstore._collection.count()} documents')

Vector store created with 2724 documents


In [12]:
retriever = vectorstore.as_retriever(search_kwargs={'k': 3}) #get top 3 closest chunks using semantic search
print('Vector store ready.')

Vector store ready.


In [13]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOllama(model='llama3.1', temperature=0)

prompt = ChatPromptTemplate.from_template("""
"You are a scholar of Gita. Answer only from the given context."
"If the answer is not in the context, say 'This is not mentioned in the text.'"

Context:
{context}

Question: {question}

Answer:
""")

def format_docs(docs):
    result = '\n\n'.join(doc.page_content for doc in docs)
    return result

# RAG chain using LCEL
rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print('RAG chain ready.')

RAG chain ready.


In [14]:
questions = ["""
    1. What does Krishna say about action without attachment?
    2. What is the nature of the soul according to the Gita?
    3. What does the Gita say about a person of steady wisdom?
    4. What are the three modes of nature?
    5. What does Krishna say about those who worship other gods?"""

]

for question in questions:
    print(f'Q: {question}')
    answer = rag_chain.invoke(question)
    print(f'A: {answer}')
    print()

Q: 
    1. What does Krishna say about action without attachment?
    2. What is the nature of the soul according to the Gita?
    3. What does the Gita say about a person of steady wisdom?
    4. What are the three modes of nature?
    5. What does Krishna say about those who worship other gods?
A: I'll answer each question based on the given context.

1. This is not mentioned in the text.
2. This is not mentioned in the text.
3. This is not mentioned in the text.
4. This is not mentioned in the text.
5. This is not mentioned in the text.

However, I can provide some answers related to the provided verses (2.28) and the context of chapters 3-5:

* Verse 2.28 mentions that living beings get a body from inert nature and it will merge back with inert nature after death, so there's no need to grieve over such a body.
* The text does not directly answer questions 1-5, but it provides some context about the chapters 3-5, which discuss dhyana yoga (path of meditation), characteristics of a t

In [10]:
question = 'What does Krishna say about those who worship other gods?'

# Without RAG
plain_answer = llm.invoke(question).content

# With RAG
rag_answer = rag_chain.invoke(question)

print('WITHOUT RAG:')
print(plain_answer)
print()
print('WITH RAG:')
print(rag_answer)

WITHOUT RAG:
In the Bhagavad Gita, a sacred Hindu scripture, Lord Krishna discusses the concept of devotion and worship. According to Krishna, there are different types of devotees and their levels of understanding.

Krishna says that even if one worships other gods or deities, they will ultimately attain spiritual growth and liberation (moksha) if their intention is pure and their heart is devoted to God. This is because the ultimate reality is the Supreme Being, who is beyond all names and forms.

In Chapter 9 of the Bhagavad Gita, Krishna says:

"Even those who worship other gods with faith will attain the goal (of liberation). For I am the only one who can grant them that goal." (Verse 23)

Krishna also emphasizes that devotion to any deity is ultimately a form of devotion to Him. He says:

"I am the source of all spiritual and material worlds. Everything emanates from Me. The wise who perfectly know this engage in worshiping Me with love." (Chapter 10, Verse 8)

In other words, Kr

1. What chunk size worked best and why?
   Initially, I used a chunk size of 500 words. While this allowed for fine-grained detail retrieval, it resulted in a larger number of chunks, which increased processing and retrieval time. To improve efficiency while still retaining sufficient context for each segment, I increased the chunk size to 800 words. This reduced the total number of chunks and sped up processing without losing coherence in the text."

2. Did the retriever ever return irrelevant chunks? Give an example.
    When I asked specific questions from the Bhagavad Gita, the retriever sometimes returned irrelevant chunks. For example, it would say there is no information for my question, but then include other unrelated content, such as verse 2.28 or general context from chapters 3–5. This additional information did not directly answer my questions, so it was unnecessary and irrelevant.
    
    
4. What would break this system? What are its limitations?


In [11]:
answer = rag_chain.invoke('2. What is the nature of the soul according to the Gita?')
print(f'A: {answer}')

A: This is not mentioned in the text. The context only discusses the characteristics of one who is steadfast in consciousness, but does not describe the nature of the soul itself.
